# SPX IV model inputs와 계산 결과 사용 가이드

이 문서는 `05_01_implied_volatility_calculation.ipynb`에 추가된 **무위험금리 `r`, forward `F`, 배당수익률 `q` 생성 과정**, enriched CSV, 행별 implied-volatility(IV) 계산, 현재 실행 결과와 후속 사용법을 설명한다.

먼저 현재 `iv_run_summary`의 의미를 분명히 하면 다음과 같다.

- 전체 424개 옵션 행 중 **400개 행은 IV 계산에 성공**했다.
- **24개 행은 IV 계산에 실패**했고 `implied_volatility`이 `NaN`으로 남았다.
- 이 24건은 solver 반복 중 수렴하지 못한 행이 아니다. 모든 실패는 root finding 전에 `mid`가 BSM 유럽형 옵션의 엄격한 할인 무차익 하한보다 낮아 `ValueError`로 거부된 경우다.
- 실패값을 IV 0이나 임의의 bracket endpoint로 대체하지 않았다. 원래 행과 실패 사유를 보존한 것이 의도된 동작이다.

관련 파일은 다음과 같다.

- 분석 Notebook: [05_01_implied_volatility_calculation.ipynb](../notebooks/05_implied_volatility/05_01_implied_volatility_calculation.ipynb)
- `F`와 `q` 계산 모듈: [model_inputs.py](../src/option_pricing_volatility/volatility/model_inputs.py)
- IV solver: [implied_volatility.py](../src/option_pricing_volatility/volatility/implied_volatility.py)
- 공개 package export: [volatility/__init__.py](../src/option_pricing_volatility/volatility/__init__.py)
- 새 계산의 단위 테스트: [test_model_inputs.py](../tests/test_model_inputs.py)
- 금융·수치 계약: [model_contracts.md](model_contracts.md)
- SPX 데이터 계약: [data_contracts.md](data_contracts.md)

## 1. 전체 데이터 흐름

<pre>
기존 processed CSV (424행 × 34열, 읽기 전용 입력)
        |
        | copy
        v
2026-07-15 DGS1MO 3.73%를 연속복리 r로 변환
        |
        +--> 동일 strike의 finite call/put mid pair 209개
                    |
                    +--> pair별 F_i 계산 --> median F
                                            |
                                            +--> q = r - log(F/S)/T
        |
        v
r/F/q 3열 + provenance 3열을 모든 행에 추가
        |
        +--> enriched CSV (424행 × 40열) 저장
        |
        +--> 메모리에서 target_price = market_price = mid 추가
                    |
                    +--> 기존 implied_volatility()를 행별 호출
                    |       |
                    |       +--> success 400 / failed 24
                    |
                    +--> log_forward_moneyness = log(K/F)
                            |
                            +--> spx_iv_df / iv_skew_df
</pre>

원본 processed CSV는 수정하거나 덮어쓰지 않는다. Enriched CSV는 모델 입력까지 포함하는 재현 가능한 중간 산출물이고, IV 결과 열은 현재 분석 Notebook의 메모리 객체 `spx_iv_df`에 추가된다. 즉, enriched CSV 자체에는 `implied_volatility`이나 `iv_status`가 들어 있지 않다는 점을 구분해야 한다.

## 2. Risk-free rate `r`

분석 snapshot의 quote date는 2026-07-15이고 실제 만기는 30일 뒤인 2026-08-14다. 

정확한 분석을 위해서는 SOFR OIS 의 데이터를 이용해서 bootstrap, interpolation등 기법을 이용해 기준일로부터 discount curve를 계산해야한다.
하지만, 본 프로젝트에서는 간단화를 위해서, FED 데이터를 사용한다. 다만, 이는 기준일에서 과거 30일의 데이터에 대한 데이터이므로 정확하지는 않다.

다만, 만기 30일은 극단적을 짧은 만기이므로 큰 차이가 없는 근사임을 적용한다.

 [New York FED](https://www.newyorkfed.org/markets/reference-rates/sofr-averages-and-index)의 **2026-07-15 backward-looking SOFR Average rate 3.62%**를 사용한다.


$$
R_{\text{SOFR}
}=0.0362
$$

$$
r = \frac{365}{30}\log\left(1 + R_{\text{SOFR}}\frac{30}{360}\right) = 0.0366785
$$



한 snapshot과 한 expiry만 분석하므로 이 값 하나를 모든 행에 사용한다. SOFR OIS bootstrap, interpolation, 별도 금리모형 또는 만기별 보간은 수행하지 않는다.

## 3. `F`와 `q` 계산 모듈의 구조

재사용 가능한 계산은 Notebook에 중복하지 않고 `volatility/model_inputs.py`의 `estimate_forward_and_dividend_yield()`에 둔다. 함수는 `DataFrame`과 연속복리 `risk_free_rate`를 받고 immutable 결과 객체를 반환한다.

```python
@dataclass(frozen=True, slots=True)
class ForwardDividendEstimate:
    forward: float
    dividend_yield: float
```

함수 내부의 처리 순서는 다음과 같다.

1. `mid`, `spot`, `T`, `strike`, `option_type` 필수 열과 finite real `risk_free_rate`를 확인한다.
2. 유효 call/put 행 전체가 하나의 양수 `spot`과 하나의 양수 `T`를 공유하는지 확인한다. `spot` 또는 `T`가 다른 행이 섞이면 하나의 `F/q`를 전체 행에 전파할 수 없으므로 실패시킨다.
3. `strike`, `mid`, `spot`, `T`가 finite이고 금융 도메인이 유효한 call/put 행만 pairing 후보로 남긴다.
4. 같은 `(strike, option_type)`이 중복되면 조용히 평균하지 않고 거부한다.
5. Call과 put을 `strike`로 one-to-one inner join한다. 한쪽 option type만 존재하는 strike와 nonfinite `mid`를 가진 pair는 forward 후보에서 제외된다.
6. 각 pair에 대해 다음 put-call parity forward 후보를 계산한다.

$$
F_i=K_i+e^{rT}(C_i-P_i)
$$

7. Finite 후보의 median을 snapshot 대표 forward로 사용한다. Mean이 아니라 median을 사용하므로 소수의 극단적인 pair가 대표값에 미치는 영향이 작다.
8. 대표 forward가 양수인지 확인한 뒤 다음 식으로 연속복리 배당수익률을 계산한다.

$$
q=r-\frac{\log(F/S)}{T}
$$

이 정의 때문에 반환된 두 값은 수치 오차 범위에서 항상 $F=S e^{(r-q)T}$를 만족한다. 회귀, 최적화, 유동성 가중치, dividend forecast는 포함하지 않는다.

## 4. Enriched CSV의 내용과 현재 값

생성 경로는 다음과 같다.

```text
data/processed/marketdata_spx/
SPX_2026-07-15_dte030_pm_sl450_model_inputs.csv
```

기존 `SPX_2026-07-15_dte030_pm_sl450_processed.csv`의 34개 열과 424개 행을 복사한 뒤 다음 6개 열만 추가한 424행 × 40열 파일이다.

| 추가 열 | 현재 값 또는 의미 |
|---|---|
| `risk_free_rate` | `0.036678460488582296` |
| `forward` | `7594.912471205` |
| `dividend_yield` | `0.0005774504688557322` |
| `risk_free_rate_source` | New york FED SOFR rate 변환식 |
| `forward_method` | 동일 strike call/put `mid` parity 후보의 median |
| `dividend_yield_method` | `q = r - log(F / S) / T` |

현재 snapshot에는 유효한 동일-strike finite call/put pair가 209개 있다. 이 값들의 median `F`를 구한 뒤 공통 `q`를 계산했다. 


## 5. Enriched 입력에서 행별 IV로 연결되는 과정

Enriched DataFrame을 만든 뒤 Notebook은 메모리에서 다음 두 열을 추가한다.

```python
spx_df["target_price"] = spx_df["mid"]
spx_df["market_price"] = spx_df["target_price"]
```

각 행의 `invert_spx_row()`는 새 입력을 기존 scalar solver에 직접 전달한다.

```python
result = implied_volatility(
    spot=row["spot"],
    strike=row["strike"],
    maturity=row["T"],
    rate=row["risk_free_rate"],
    market_price=row["market_price"],
    option_type=row["option_type"],
    dividend_yield=row["dividend_yield"],
)
```

Solver의 BSM 공식이나 bisection 로직은 수정하거나 Notebook에 재구현하지 않는다. 성공하면 IV, signed repricing error, 반복 횟수와 수렴 상태를 저장한다. `ValueError` 또는 `RuntimeError`가 발생하면 행을 삭제하지 않고 다음처럼 명시적으로 기록한다.

| 결과 열 | 성공 행 | 실패 행 |
|---|---|---|
| `implied_volatility` | 계산된 IV | `NaN` |
| `repricing_error` | `BSM price - mid` | `NaN` |
| `iv_iterations` | bisection 반복 횟수 | `NaN` |
| `iv_converged` | `True` | `False` |
| `iv_status` | `success` | `failed` |
| `iv_failure_reason` | 결측 | 예외 타입과 메시지 |

마지막으로 모든 행에 공통 forward를 사용해 다음 skew 좌표를 계산한다.

$$
\text{log forward moneyness}=\log(K/F)
$$

이 값은 strike 자체보다 forward 대비 상대 위치를 나타내므로 이후 IV skew의 x축으로 사용한다.

## 6. `iv_run_summary`와 24개 실패 행의 정확한 의미

현재 전체 상태만 집계하면 다음과 같다.

| `iv_status` | 행 수 | 비율 |
|---|---:|---:|
| `success` | 400 | 94.34% |
| `failed` | 24 | 5.66% |
| 합계 | 424 | 100.00% |

따라서 질문의 해석처럼 **IV 계산에 실패한 원본 옵션 행이 총 24개**라는 의미가 맞다. 다만 화면의 `iv_run_summary`에서 failed가 24줄로 나뉘고 각 `row_count`가 1인 이유는 다음 groupby가 전체 오류 문자열까지 key로 사용하기 때문이다.

```python
spx_iv_df.groupby(["iv_status", "iv_failure_reason"], dropna=False).size()
```

무차익 오류 메시지에는 각 행마다 다른 lower bound, upper bound와 `market_price` 숫자가 들어간다. 따라서 같은 종류의 오류라도 문자열이 서로 달라 24개 그룹으로 분리된다. 상태별 총 행 수만 확인할 때는 다음 집계가 더 직접적이다.

```python
spx_iv_df["iv_status"].value_counts(dropna=False)
# success    400
# failed      24
```

현재 24개는 모두 다음 한 종류다.

| option type | 실패 수 | strike | 공통 판정 |
|---|---:|---|---|
| call | 16 | 2400, 2600, 2800, 3000, 3200, 3400, 3600, 3800, 4000, 4200, 4400, 4600, 4800, 5000, 5200, 5800 | `mid < call lower bound` |
| put | 8 | 8400, 8500, 8800, 9000, 9200, 9400, 9600, 9800 | `mid < put lower bound` |

이 중 22개는 기존 `analysis_candidate=False`이고, 8400 put과 8500 put 두 행만 `analysis_candidate=True`다. 따라서 단순히 `analysis_candidate`만 필터링해도 모든 invalid IV가 제거되는 것은 아니며 반드시 `iv_status == "success"`도 함께 확인해야 한다.

## 7. 왜 이 행들에서는 IV가 존재하지 않는가

주어진 `S`, `K`, `T`, `r`, `q` 아래 유럽형 call과 put의 할인 무차익 범위는 다음과 같다.

$$
\max(0,S e^{-qT}-K e^{-rT}) \le C \le S e^{-qT}
$$

$$
\max(0,K e^{-rT}-S e^{-qT}) \le P \le K e^{-rT}
$$

24개 실패 행의 `mid`는 모두 해당 option type의 **lower bound보다 낮다**. 차이는 약 0.0748에서 3.3129 index point다. Lower bound는 `sigma=0`에서도 가능한 최소 BSM 가격이므로, 이보다 낮은 target은 어떤 음이 아닌 변동성을 넣어도 BSM으로 재현할 수 없다. 따라서 bisection을 더 오래 실행하거나 volatility bracket을 넓혀도 해결되지 않는다.

현재 데이터에서는 24행 모두 `bid < theoretical lower bound < ask`이며, 거래량은 모두 0이다. Open interest도 23행은 0, 한 행은 1이다. 즉, midpoint는 하한보다 조금 낮지만 매수 실행가격인 ask는 하한 위에 있다.

이 결과만으로 특정 원인을 단정해서는 안 된다. 가능한 설명에는 deep-ITM 구간의 quote 비동시성 또는 stale quote, midpoint와 공통 `S/r/q/T` 사이의 작은 불일치, bid/ask 단위의 시장 미세구조 잡음, 하나의 median `F/q`를 전체 strike에 적용하는 단순화가 있다. 이는 **현재 입력과 BSM 가정의 조합으로 해당 midpoint를 설명할 수 없다는 진단**이지, 반드시 실행 가능한 시장 arbitrage가 존재한다는 결론이나 solver 버그라는 뜻은 아니다.

프로젝트 계약은 무차익 범위를 tolerance만큼 넓히거나 target을 lower bound로 clipping하는 것을 금지한다. 따라서 현재처럼 실패 사유를 남기고 IV를 `NaN`으로 유지하는 것이 올바른 처리다.

## 8. 후속 분석에서 결과물을 사용하는 방법

### 8.1 모델 입력이 필요한 다음 Notebook

다른 분석에서 같은 snapshot의 BSM 입력이 필요하면 원본 CSV가 아니라 enriched CSV를 읽는다. 이 파일에는 출처가 있는 공통 `r/F/q`가 이미 포함되어 있다. `target_price`가 필요하면 데이터 계약대로 `mid`에서 다시 만든다.

```python
model_inputs_df = pd.read_csv(SPX_MODEL_INPUTS_PATH)
model_inputs_df["target_price"] = model_inputs_df["mid"]
```

Enriched CSV의 `risk_free_rate_source`, `forward_method`, `dividend_yield_method`도 최종 표나 그림의 재현성 기록과 함께 보존한다.

### 8.2 IV skew 표본

IV plot, 요약통계 또는 ATM 선택에는 성공 행만 사용한다. 실패 IV를 0으로 대체하거나 forward-fill하지 않는다. 기존 quality filter까지 적용하려면 다음 조건을 함께 사용한다.

```python
skew_input = spx_iv_df.loc[
    spx_iv_df["iv_status"].eq("success")
    & spx_iv_df["analysis_candidate"]
    & np.isfinite(spx_iv_df["log_forward_moneyness"])
].copy()
```

현재 결과에서는 `analysis_candidate=True`인 352행 중 350행이 IV success이고 2행이 failed다. 위 조건은 그 두 행까지 안전하게 제외한다. 이후 call/put을 구분해 `log_forward_moneyness`를 x축, `implied_volatility`를 y축으로 사용하면 된다.

### 8.3 실패 행의 품질 점검

실패 행은 삭제 대상이라기보다 진단 표본이다. 별도 표에서 다음 열을 함께 확인한다.

```python
failed_iv_rows = spx_iv_df.loc[
    spx_iv_df["iv_status"].eq("failed"),
    [
        "option_symbol", "option_type", "strike",
        "bid", "ask", "mid", "relative_spread",
        "risk_free_rate", "forward", "dividend_yield",
        "iv_failure_reason",
    ],
]
```

최종 분석에서는 `400 success / 24 failed`와 사용한 추가 quality filter를 함께 보고해야 한다. 그래야 skew가 전체 424행이 아니라 유효 IV subset에서 만들어졌음을 알 수 있다.

## 9. 테스트, 검증 결과와 현재 한계

`tests/test_model_inputs.py`가 직접 보호하는 동작은 다음과 같다.

- Synthetic parity 후보에서 median `F`와 식에 맞는 `q`를 복원한다.
- Pair가 없는 strike와 nonfinite `mid` pair는 후보에서 제외한다.
- 유효한 동일-strike call/put pair가 하나도 없으면 `ValueError`를 발생시킨다.
- 다른 `spot` 또는 `T`의 행이 섞이면 하나의 `F/q`를 잘못 broadcast하지 않고 거부한다.

구현 완료 후 확인한 결과는 다음과 같다.

| 검증 | 결과 |
|---|---|
| 전체 `python -m pytest` | 92 passed |
| Notebook top-to-bottom 실행 | finance Python 3.12에서 완료 |
| 원본 대 enriched 비교 | 원본 파일 불변, 원본 34열 schema와 424행 보존 |
| 공통 model input | 모든 행에서 `r/F/q` 각각 한 값 |
| Forward identity | `F = S exp((r-q)T)` 일치 |
| `log(K/F)` | 424행 모두 finite |
| IV 결과 | 400 success, 24 explicit failure |
| 성공행 최대 절대 repricing error | 약 `9.98e-9` |

현재 방법의 의도적인 한계는 다음과 같다.

- 단일 snapshot과 단일 expiry만 지원하며 만기 구조를 만들지 않는다.
- DGS1MO 한 점을 사용하며 zero-coupon curve나 interpolation을 구현하지 않는다.
- 대표 forward는 parity 후보의 단순 median이며 spread 또는 liquidity 가중치를 사용하지 않는다.
- 하나의 implied `q`를 전체 strike에 사용하며 dividend forecast를 수행하지 않는다.
- IV 실패를 자동 수정하거나 대체값으로 채우지 않는다.
- `spx_iv_df`와 `iv_skew_df`는 현재 Notebook 메모리 산출물이며 별도 IV 결과 CSV로 저장하지 않는다.

이 범위를 유지하면 데이터 provenance, model input 생성, IV inversion과 분석 표본 선택의 책임이 분리되고, 실패 행도 추적 가능한 상태로 남는다.